# 05 — Two agents, sequential

**Goal.** Split agent work into two roles that talk through DuckDB, not through a shared message list:

- **Collector** — given a high-level intent ("check what's new"), decide what to fetch and trigger it.
- **Analyst** — read what the collector produced, classify-and-summarise, decide whether to raise an alert.

Each agent is small, has a tight system prompt, and only sees the tools relevant to its job. They never share working memory.

**Why sequential and not concurrent.** On a single laptop with one 8 GB GPU, async multi-agent introduces orchestration complexity that obscures the lesson. Sequential = easy to log, easy to debug, easy to extend. Production systems can be made concurrent later; the *separation of concerns* is what carries over.

**Why two agents instead of one.** A single agent with too many tools tends to wander. Local 8B models are particularly prone to it. Two specialists, each with 3–4 tools, behave noticeably better.

## 1. Setup

We re-import nb 01 (read-only tools) and nb 04 (memory + writable connection). The two helpers we need are `dispatch_tool_call` and the writable `RW_CON`.

In [ ]:
import json, nbformat, ollama
from pathlib import Path
from datetime import datetime, timezone
import duckdb

for nb_path in ["01_tools_not_agents.ipynb", "04_memory.ipynb"]:
    nb = nbformat.read(Path(nb_path), as_version=4)
    for cell in nb.cells:
        if cell.cell_type == "code":
            exec(cell.source, globals())

MODEL = "llama3.1:8b"
print("setup done. tools:", [t['function']['name'] for t in TOOL_SCHEMAS])

## 2. The handoff: a tiny task queue in DuckDB

The collector writes rows; the analyst reads them. That's the entire "protocol". One table, two states (`pending` → `done`).

In [ ]:
RW_CON.execute("""
CREATE TABLE IF NOT EXISTS agent_tasks (
    task_id     INTEGER PRIMARY KEY,
    kind        TEXT,         -- 'review_document' | 'review_recent_alerts'
    payload     TEXT,         -- JSON, e.g. {"url_id": "..."}
    state       TEXT,         -- 'pending' | 'done'
    created_at  TIMESTAMP,
    finished_at TIMESTAMP,
    result      TEXT          -- analyst's verdict, free text
);
""")
RW_CON.execute("CREATE SEQUENCE IF NOT EXISTS agent_tasks_seq START 1;")
print("agent_tasks table ready")

## 3. Collector agent

Tools the collector sees: `count_recent_victims`, `query_documents`, `query_alerts`, plus a *new* tool — `enqueue_task` — that lets it ask the analyst to look at something. It does **not** see `get_document` or any memory tools — it shouldn't be reading bodies or writing notes; that's the analyst's job.

In [ ]:
ENQUEUE_TASK_SCHEMA = {
    "name": "enqueue_task",
    "description": (
        "Hand a task to the analyst agent. kind must be 'review_document' (with payload "
        "{'url_id': ...}) or 'review_recent_alerts' (with payload {'since_days': N})."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "kind": {"type": "string", "enum": ["review_document", "review_recent_alerts"]},
            "payload": {"type": "object", "description": "Task-specific arguments."},
        },
        "required": ["kind", "payload"],
    },
}

def enqueue_task(kind, payload):
    if kind not in ("review_document", "review_recent_alerts"):
        return {"error": f"unknown kind: {kind}"}
    if not isinstance(payload, dict):
        return {"error": "payload must be an object"}
    tid = RW_CON.execute("SELECT nextval('agent_tasks_seq')").fetchone()[0]
    RW_CON.execute(
        "INSERT INTO agent_tasks VALUES (?, ?, ?, 'pending', ?, NULL, NULL)",
        [tid, kind, json.dumps(payload), datetime.now(timezone.utc)],
    )
    return {"ok": True, "task_id": tid}

# Per-agent tool registries — the key idea of nb 05.
COLLECTOR_TOOLS = [
    {"type": "function", "function": COUNT_RECENT_VICTIMS_SCHEMA},
    {"type": "function", "function": QUERY_DOCUMENTS_SCHEMA},
    {"type": "function", "function": QUERY_ALERTS_SCHEMA},
    {"type": "function", "function": ENQUEUE_TASK_SCHEMA},
]
COLLECTOR_FN_NAMES = {"count_recent_victims", "query_documents", "query_alerts", "enqueue_task"}

TOOL_FUNCTIONS["enqueue_task"] = enqueue_task

COLLECTOR_PROMPT = (
    "You are the COLLECTOR agent in a CTI monitoring pipeline.\n"
    "Your job: scan what is new, decide what is interesting enough for the analyst "
    "to review, and enqueue tasks for them via enqueue_task. You do not read documents "
    "yourself.\n\n"
    "Rules:\n"
    "- Enqueue at most 3 tasks per run. Be selective.\n"
    "- Use kind='review_document' with a real url_id from query_documents results.\n"
    "- Use kind='review_recent_alerts' if there are unreviewed high-severity alerts.\n"
    "- When done, briefly state what you enqueued and stop."
)

## 4. Analyst agent

Tools the analyst sees: `query_documents`, `get_document`, `query_alerts`, `remember_note`, plus `complete_task` so it can mark its work done. It does **not** see `enqueue_task` — only the collector creates work.

In [ ]:
COMPLETE_TASK_SCHEMA = {
    "name": "complete_task",
    "description": "Mark a pending task as done and record your verdict.",
    "parameters": {
        "type": "object",
        "properties": {
            "task_id": {"type": "integer"},
            "result": {"type": "string", "description": "Short verdict / summary."},
        },
        "required": ["task_id", "result"],
    },
}

def complete_task(task_id, result):
    n = RW_CON.execute(
        "UPDATE agent_tasks SET state='done', finished_at=?, result=? "
        "WHERE task_id=? AND state='pending'",
        [datetime.now(timezone.utc), (result or "")[:1000], int(task_id)],
    ).fetchone()
    return {"ok": True, "task_id": int(task_id)}

ANALYST_TOOLS = [
    {"type": "function", "function": QUERY_DOCUMENTS_SCHEMA},
    {"type": "function", "function": GET_DOCUMENT_SCHEMA},
    {"type": "function", "function": QUERY_ALERTS_SCHEMA},
    {"type": "function", "function": REMEMBER_NOTE_SCHEMA},
    {"type": "function", "function": COMPLETE_TASK_SCHEMA},
]
TOOL_FUNCTIONS["complete_task"] = complete_task

ANALYST_PROMPT = (
    "You are the ANALYST agent in a CTI monitoring pipeline.\n"
    "You are given one task at a time. For each task: read what is needed, form a "
    "short verdict, optionally save a note to long-term memory, then call complete_task.\n\n"
    "Rules:\n"
    "- Always call complete_task at the end. The task is not done until you do.\n"
    "- Keep verdicts to 1-3 sentences.\n"
    "- Only call remember_note for genuinely useful long-term facts."
)

## 5. A small loop runner

Generic enough to drive both agents. The only difference between collector and analyst is the system prompt and the tool list it sees — proof that the loop machinery from nb 03 is the agentic primitive.

In [ ]:
def truncate_for_model(payload, max_chars=1500):
    s = json.dumps(payload)
    return s if len(s) <= max_chars else s[:max_chars] + f"... [truncated, total {len(s)}]"

def run_specialist(system_prompt, tools_list, user_msg, *, label="agent",
                   max_steps=4, max_tool_calls=6, model=MODEL, verbose=True):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_msg},
    ]
    total_calls = 0
    for step in range(1, max_steps + 1):
        if verbose: print(f"\n--- {label} step {step} ---")
        resp = ollama.chat(model=model, messages=messages, tools=tools_list,
                           options={"temperature": 0.2, "num_ctx": 4096})
        msg = resp["message"]
        messages.append(msg)
        tool_calls = msg.get("tool_calls") or []
        if not tool_calls:
            if verbose:
                print(f"[{label}] final:", (msg.get('content') or '')[:300])
            return msg.get("content") or ""
        for tc in tool_calls:
            fn_name = tc["function"]["name"]
            fn_args = tc["function"].get("arguments") or {}
            if isinstance(fn_args, str):
                try: fn_args = json.loads(fn_args)
                except json.JSONDecodeError: fn_args = {}
            total_calls += 1
            if total_calls > max_tool_calls:
                if verbose: print(f"[{label}] tool-call cap hit")
                return f"({label}: stopped at tool cap)"
            result = dispatch_tool_call(fn_name, fn_args)
            content = truncate_for_model(result)
            messages.append({"role": "tool", "name": fn_name, "content": content})
            if verbose:
                print(f"  -> {fn_name}({fn_args})")
                print(f"     {content[:200]}")
    if verbose: print(f"[{label}] step cap hit")
    return f"({label}: stopped at step cap)"

## 6. Run the pipeline

Three phases:

1. **Collector** scans, decides, enqueues.
2. **Dispatcher** (just Python) reads pending tasks one at a time.
3. **Analyst** processes each task, completes it.

We run the dispatcher manually as a `for` loop. In production this would be a daemon, a cron job, or — eventually — async.

In [ ]:
# Phase 1: collector
_ = run_specialist(
    COLLECTOR_PROMPT,
    COLLECTOR_TOOLS,
    "Scan the corpus and the ransomware feed. Enqueue 1-2 review tasks for the analyst.",
    label="collector",
)

pending = RW_CON.execute(
    "SELECT task_id, kind, payload FROM agent_tasks WHERE state='pending' ORDER BY task_id"
).fetchall()
print(f"\npending tasks: {len(pending)}")
for t in pending:
    print(" ", t)

In [ ]:
# Phases 2 + 3: dispatcher hands tasks to the analyst, one at a time.
for task_id, kind, payload_json in pending:
    print(f"\n############# task {task_id} ({kind}) #############")
    user_msg = (
        f"You have task_id={task_id}, kind='{kind}', payload={payload_json}.\n"
        f"Investigate and call complete_task with your verdict."
    )
    _ = run_specialist(ANALYST_PROMPT, ANALYST_TOOLS, user_msg, label=f"analyst#{task_id}")

print("\n=== completed tasks ===")
for r in RW_CON.execute("SELECT task_id, kind, state, result FROM agent_tasks ORDER BY task_id").fetchall():
    print(" ", r)

print("\n=== notes the analyst saved ===")
for n in recall_notes(limit=10)["notes"]:
    print(" ", n)

## 7. What this teaches

- **Separation of concerns is real.** Two agents with 4–5 tools each behave better than one agent with 9 tools.
- **State lives in the database, not the message list.** Either agent can crash and restart; the task queue and notes survive.
- **Sequential is fine for teaching.** All the lessons of multi-agent — handoff design, role isolation, partial failure — show up here. Concurrency is a deployment concern, not a learning one.

## What's next

nb 06 — evaluation and guardrails. We'll record traces of every model call, replay them without re-spending compute, count failure modes, add a budget cap (steps and tool calls per run), and a human-in-the-loop confirmation pattern for tools that mutate state.